# 03 - Fraud Pattern Analysis

This notebook performs in-depth fraud pattern detection and analysis.

## Objectives
- Detect fraud patterns across all entities
- Analyze regional and temporal fraud patterns
- Generate risk scores and rankings
- Produce actionable fraud reports

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import sys
sys.path.insert(0, '..')

from src.etl.extractors import extract_all
from src.etl.transformers import transform_all
from src.features.driver_features import create_driver_features, get_driver_risk_scores
from src.features.customer_features import create_customer_features, get_customer_risk_scores
from src.analysis.fraud_patterns import (
    analyze_all_fraud_patterns, 
    generate_fraud_report,
    detect_collusion_patterns
)
from src.analysis.geographic import analyze_regional_performance, identify_regional_hotspots
from src.analysis.temporal import get_temporal_summary, detect_temporal_anomalies

pd.set_option('display.max_columns', None)
%matplotlib inline

In [ ]:
# Load and transform all data
raw_data = extract_all()
data = transform_all(raw_data)

orders = data['orders']
drivers = data['drivers']
customers = data['customers']
products = data['products']
missing_items = data['missing_items']

print("Data loaded successfully!")

## 1. Overall Fraud Metrics

In [ ]:
# Calculate overall metrics
total_orders = len(orders)
total_items = orders['items_delivered'].sum() + orders['items_missing'].sum()
total_missing = orders['items_missing'].sum()
orders_with_missing = (orders['items_missing'] > 0).sum()

print("=" * 50)
print("OVERALL FRAUD METRICS")
print("=" * 50)
print(f"Total Orders: {total_orders:,}")
print(f"Total Items: {total_items:,}")
print(f"Total Missing Items: {total_missing:,}")
print(f"Overall Missing Rate: {total_missing/total_items*100:.2f}%")
print(f"Orders with Missing Items: {orders_with_missing:,} ({orders_with_missing/total_orders*100:.2f}%)")
print(f"Estimated Loss (at $10/item): ${total_missing * 10:,.2f}")

## 2. Comprehensive Fraud Pattern Detection

In [ ]:
# Run all fraud pattern analyses
fraud_indicators = analyze_all_fraud_patterns(orders, drivers, customers)

print("Fraud Indicators Found:")
for pattern_type, indicators in fraud_indicators.items():
    print(f"  - {pattern_type}: {len(indicators)} indicators")

In [ ]:
# Generate fraud report
fraud_report = generate_fraud_report(fraud_indicators)

print(f"\nTotal Fraud Indicators: {fraud_report['summary']['total_indicators']}")
print(f"High Risk: {len(fraud_report['high_risk'])}")
print(f"Medium Risk: {len(fraud_report['medium_risk'])}")
print(f"Low Risk: {len(fraud_report['low_risk'])}")

In [ ]:
# Display high-risk entities
if fraud_report['high_risk']:
    high_risk_df = pd.DataFrame(fraud_report['high_risk'])
    display(high_risk_df)

## 3. Driver Risk Analysis

In [ ]:
# Create driver features and risk scores
driver_features = create_driver_features(drivers, orders)
driver_risk = get_driver_risk_scores(driver_features)

In [ ]:
# Driver risk distribution
fig = px.histogram(
    driver_risk,
    x='risk_score',
    color='risk_category',
    nbins=20,
    title='Driver Risk Score Distribution',
    labels={'risk_score': 'Risk Score', 'risk_category': 'Risk Category'}
)
fig.show()

In [ ]:
# Top 10 highest risk drivers
top_risk_drivers = driver_risk.nlargest(10, 'risk_score')[[
    'driver_id', 'driver_name', 'age', 'total_orders', 
    'missing_rate', 'risk_score', 'risk_category'
]]
display(top_risk_drivers)

In [ ]:
# Risk category breakdown
risk_counts = driver_risk['risk_category'].value_counts()
fig = px.pie(
    values=risk_counts.values,
    names=risk_counts.index,
    title='Driver Risk Category Distribution',
    color_discrete_sequence=px.colors.sequential.RdBu
)
fig.show()

## 4. Customer Risk Analysis

In [ ]:
# Create customer features and risk scores
customer_features = create_customer_features(customers, orders)
customer_risk = get_customer_risk_scores(customer_features)

In [ ]:
# Customer risk distribution
fig = px.histogram(
    customer_risk[customer_risk['total_orders'] > 0],
    x='risk_score',
    color='risk_category',
    nbins=20,
    title='Customer Risk Score Distribution',
    labels={'risk_score': 'Risk Score', 'risk_category': 'Risk Category'}
)
fig.show()

In [ ]:
# Top 10 highest risk customers
top_risk_customers = customer_risk[customer_risk['total_orders'] >= 2].nlargest(10, 'risk_score')[[
    'customer_id', 'customer_name', 'customer_age', 'total_orders',
    'missing_rate', 'total_spent', 'risk_score', 'risk_category'
]]
display(top_risk_customers)

## 5. Regional Fraud Hotspots

In [ ]:
# Regional performance
regional = analyze_regional_performance(orders)
regional.sort_values('missing_rate', ascending=False)

In [ ]:
# Identify hotspots
hotspots = identify_regional_hotspots(orders, threshold_pct=75)

print("Regional Hotspots:")
for category, regions in hotspots.items():
    print(f"  {category}: {regions}")

In [ ]:
# Regional missing rate heatmap
fig = px.bar(
    regional.sort_values('missing_rate', ascending=True),
    x='missing_rate',
    y='region',
    orientation='h',
    color='missing_rate',
    color_continuous_scale='Reds',
    title='Missing Rate by Region',
    labels={'missing_rate': 'Missing Rate (%)', 'region': 'Region'}
)
fig.add_vline(
    x=regional['missing_rate'].mean(),
    line_dash='dash',
    line_color='blue',
    annotation_text='Average'
)
fig.show()

## 6. Temporal Fraud Patterns

In [ ]:
# Temporal summary
temporal_summary = get_temporal_summary(orders)

print("Temporal Fraud Analysis:")
print(f"  Trend Direction: {temporal_summary['trend']['direction']}")
print(f"  Peak Month: {temporal_summary['trend']['peak_month']} ({temporal_summary['trend']['peak_rate']:.2f}%)")
print(f"  Worst Day: {temporal_summary['patterns']['worst_day']}")
print(f"  Worst Hour: {temporal_summary['patterns']['worst_hour']}:00")

In [ ]:
# Temporal anomalies
anomalies = detect_temporal_anomalies(orders)

print("\nTemporal Anomalies Detected:")
for anomaly_type, items in anomalies.items():
    if items:
        print(f"  {anomaly_type}: {items}")

## 7. Collusion Analysis

In [ ]:
# Detect collusion patterns
collusion_indicators = detect_collusion_patterns(orders, min_interactions=2, threshold_rate=40)

print(f"Potential Collusion Cases Found: {len(collusion_indicators)}")

if collusion_indicators:
    collusion_df = pd.DataFrame([{
        'driver_id': i.details['driver_id'],
        'customer_id': i.details['customer_id'],
        'interactions': i.details['interactions'],
        'missing_rate': i.details['missing_rate'],
        'items_missing': i.details['items_missing'],
        'risk_score': i.score
    } for i in collusion_indicators])
    display(collusion_df.head(10))

## 8. Executive Summary

### Key Findings

| Metric | Value |
|--------|-------|
| Overall Missing Rate | [Fill] |
| Orders with Issues | [Fill] |
| High-Risk Drivers | [Fill] |
| High-Risk Customers | [Fill] |
| Worst Region | [Fill] |
| Potential Collusion Cases | [Fill] |

### Recommendations
1. [Add recommendations based on analysis]
2. [Add recommendations based on analysis]
3. [Add recommendations based on analysis]